In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline



In [ ]:
df = pd.read_csv("dados_usados/dados_para_modelo.csv")

In [ ]:
df.head(10)

,status_do_emprestimo,valor_do_emprestimo,prazo_do_emprestimo_meses,taxa_de_juros,valor_da_parcela,tempo_de_emprego_anos,renda_anual,data_de_emissao_do_emprestimo,relacao_divida_renda,primeira_linha_de_credito,...,total_de_contas_de_credito,contas_de_hipoteca,registros_de_falencia,classificacao_de_risco,tipo_de_moradia,finalidade_do_emprestimo,tipo_de_aplicacao,taxa_de_juros_agrupada,taxa/falencia,juros/contas
0,0,10000.0,36,11.44,329.48,10.0,117000.0,2015,26.24,1990,...,25.0,0.0,0.0,0.874270,0.773378,0.810767,0.803913,0.25,11.44,64.0
1,0,8000.0,36,11.99,265.68,4.0,65000.0,2015,22.05,2004,...,27.0,3.0,0.0,0.874270,0.830439,0.792586,0.803913,0.50,11.99,34.0
2,0,15600.0,36,10.49,506.97,0.0,43057.0,2015,12.79,2007,...,26.0,0.0,0.0,0.874270,0.773378,0.832882,0.803913,0.25,10.49,52.0
3,0,7200.0,36,6.49,220.65,6.0,54000.0,2014,2.60,2006,...,13.0,0.0,0.0,0.937121,0.773378,0.832882,0.803913,0.25,6.49,24.0
4,1,24375.0,60,17.27,609.33,9.0,55000.0,2013,33.95,1999,...,43.0,1.0,0.0,0.788191,0.830439,0.832882,0.803913,0.50,17.27,26.0
5,0,20000.0,36,13.33,677.07,10.0,86788.0,2015,16.31,2005,...,23.0,4.0,0.0,0.788191,0.830439,0.792586,0.803913,0.50,13.33,16.0
6,0,18000.0,36,5.32,542.07,2.0,125000.0,2015,1.36,2005,...,25.0,3.0,0.0,0.937121,0.830439,0.829921,0.803913,0.25,5.32,32.0
7,0,13000.0,36,11.14,426.47,10.0,46000.0,2012,26.87,1994,...,15.0,0.0,0.0,0.874270,0.773378,0.832882,0.803913,0.25,11.14,44.0
8,0,18900.0,60,10.99,410.84,10.0,103000.0,2014,12.52,1994,...,40.0,3.0,0.0,0.874270,0.773378,0.792586,0.803913,0.25,10.99,52.0
9,0,26300.0,36,16.29,928.40,3.0,115000.0,2012,23.69,1997,...,37.0,1.0,0.0,0.788191,0.830439,0.792586,0.803913,0.50,16.29,26.0


In [ ]:
x = df.drop("status_do_emprestimo", axis=1)
y = df["status_do_emprestimo"]

In [ ]:
x.info()

<class 'pandas.DataFrame'>
RangeIndex: 383421 entries, 0 to 383420
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   valor_do_emprestimo             383421 non-null  float64
 1   prazo_do_emprestimo_meses       383421 non-null  int64  
 2   taxa_de_juros                   383421 non-null  float64
 3   valor_da_parcela                383421 non-null  float64
 4   tempo_de_emprego_anos           383421 non-null  float64
 5   renda_anual                     383421 non-null  float64
 6   data_de_emissao_do_emprestimo   383421 non-null  int64  
 7   relacao_divida_renda            383421 non-null  float64
 8   primeira_linha_de_credito       383421 non-null  int64  
 9   contas_de_credito_abertas       383421 non-null  float64
 10  registros_publicos_negativos    383421 non-null  float64
 11  saldo_de_credito_rotativo       383421 non-null  float64
 12  utilizacao_do_credito_rotat

In [ ]:
y.value_counts()

status_do_emprestimo
0    308372
1     75049
Name: count, dtype: int64

In [ ]:
x_train, x_test, y_train, y_test= train_test_split(x, y, test_size=0.2, stratify= y, random_state=42)

In [ ]:
len(x_train), len(y_train)

(306736, 306736)

In [ ]:
len(x_test), len(y_test)

(76685, 76685)

In [ ]:
pipeline_logistic = Pipeline(
   [ ("scaler", StandardScaler()),
    ("model", LogisticRegression())]
)

pipeline_knn = Pipeline(
    [("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())]
)

In [ ]:
modelos = {
    "KNN" : pipeline_knn,
    "Logistic Regressionr" : pipeline_logistic,
    "Ramdom Forest" : RandomForestClassifier(random_state=42),
    "XGBoosting" : XGBClassifier(random_state=42),
    "Lightgbm" : LGBMClassifier(random_state=42)
}

In [ ]:
avaliacao = {}

for name, modelo in modelos.items():
    cross_score = cross_val_score(
        modelo, x_train, y_train,
        cv=5,
        scoring= "roc_auc"
    )
    avaliacao[name] = np.mean(cross_score)
    print(f"{name} : {avaliacao[name]}")


In [ ]:
knn_params = {
  "model__n_neighbors" : [1, 3, 5, 7, 9, 20, 30, 50],
  "model__weights" : ["uniform", "distance"]
}

random_forest_params = {
    "n_estimators" : np.arange(300, 1500, 90),
    "max_depth" : [1, 5, 9, 20, 50],
    "min_samples_leaf" : np.arange(1, 10, 3),
    "min_samples_split" : np.arange(1, 15, 3)
}

logistic_params = {
    "model__C" : np.logspace(-3, 3, 10),
    "model__solver" : ["lbfgs", "liblinear"]
}

gradient_boosting_params = {
    "n_estimators" : np.arange(100, 1080, 80),
    "max_depth" : [1, 3, 9, 15],
    "subsample" : [0.2, 0.5, 0.7, 1],
    "min_child_weight" : np.arange(1, 10, 3),
    "scale_pos_weight" : [10, 20, 5, 15, ]
}

light_params = {
    "num_leaves" : [63, 127],
    "n_estimators" : [100, 300, 500, 1000],
    "min_child_sample" : [100, 300, 500, 1000],
    "max_depth" : [1, 5, 7],
    "subsample" : [0.2, 0.5, 1],
    "is_unbalance" : [True],
    "colsample_bytree" : [0.2, 1],
    "learning_rate" : [0.2, 1],
    "force_row_wise" : [True]
}



: 

: 

: 

: 

: 

In [ ]:
x_sample,_, y_sample,_ = train_test_split(x_train, y_train, train_size=100000, stratify=y_train, random_state=42)

: 

: 

: 

: 

: 

In [ ]:
def test_randomized_search(model, params, x_sample, y_sample):
    randomized = HalvingRandomSearchCV(
    model,
    params,
    cv= 5,
    n_candidates = 100,
    factor=3,
    scoring="roc_auc",
    n_jobs = -1,
    min_resources= 3000,
    random_state=42,
    verbose=2
)

    randomized.fit(x_sample, y_sample)
    randomized
    print(f"O melhor score foi: {randomized.best_score_}")
    print(f"Os melhores parametros foram: {randomized.best_params_}")


: 

: 

: 

: 

: 

In [ ]:
knn_modelo = test_randomized_search(pipeline_knn, knn_params, x_train, y_train)

n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 5
min_resources_: 3000
max_resources_: 306736
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 16
n_resources: 3000
Fitting 5 folds for each of 16 candidates, totalling 80 fits


/home/Erick/miniconda3/envs/data/lib/python3.11/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 16 is smaller than n_iter=100. Running 16 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.1s
[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.1s
[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.1s
[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.1s
[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.1s
[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.1s
[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.1s
[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.1s
[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.0s
[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.0s
[CV] END .......model__n_neighbors=3, model__weights=uniform; total time=   0.0s
[CV] END .......model__n_neighbors=3, model__weights=uniform; total time=   0.1s
[CV] END .......model__n_nei

: 

: 

: 

: 

: 

In [ ]:
logistic_modelo = test_randomized_search(pipeline_logistic, logistic_params, x_train, y_train)

n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 5
min_resources_: 3000
max_resources_: 306736
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 20
n_resources: 3000
Fitting 5 folds for each of 20 candidates, totalling 100 fits


/home/Erick/miniconda3/envs/data/lib/python3.11/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 20 is smaller than n_iter=100. Running 20 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.0s
[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.0s
[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.0s
[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.0s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.0s
[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.0s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.0s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.0s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.0s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.0s
[CV] END .model__C=0.004641588833612777, model__solver=lbfgs; total time=   0.0s
[CV] END .model__C=0.004641588833612777, model__solver=lbfgs; total time=   0.0s
[CV] END .model__C=0.0046415

: 

: 

: 

: 

: 

In [ ]:
gradient_modelo = test_randomized_search(XGBClassifier(), gradient_boosting_params, x_train, y_train)

n_iterations: 5
n_required_iterations: 5
n_possible_iterations: 5
min_resources_: 3000
max_resources_: 306736
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 100
n_resources: 3000
Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END max_depth=15, min_child_weight=7, n_estimators=100, scale_pos_weight=10, subsample=1; total time=   0.8s
[CV] END max_depth=15, min_child_weight=7, n_estimators=100, scale_pos_weight=10, subsample=1; total time=   0.8s
[CV] END max_depth=15, min_child_weight=7, n_estimators=100, scale_pos_weight=10, subsample=1; total time=   0.9s
[CV] END max_depth=15, min_child_weight=7, n_estimators=100, scale_pos_weight=10, subsample=1; total time=   1.0s
[CV] END max_depth=15, min_child_weight=7, n_estimators=100, scale_pos_weight=10, subsample=1; total time=   1.0s
[CV] END max_depth=3, min_child_weight=4, n_estimators=420, scale_pos_weight=20, subsample=0.7; total time=   0.7s
[CV] END max_depth=3, min_child_weight=4, n_est

: 

: 

In [ ]:
light_modelo = test_randomized_search(LGBMClassifier(), light_params, x_sample, y_sample)

NameError: name 'test_randomized_search' is not defined

: 

: 

In [ ]:
random_forest_modelo = test_randomized_search(RandomForestClassifier(), random_forest_params, x_sample, y_sample)

: 

: 

In [ ]:
best_modelo = light_modelo

: 

: 

In [ ]:
best_modelo.best_params_

: 

: 

In [ ]:
best_modelo.feature_importances_

: 

: 